# Goal: statistical-summary 

In [13]:
""" 
Goal: statistical-summary
Author: Rudra Prasad Bhuyan
Date: 10-10-2026 14:35 IST
"""

' \nGoal: statistical-summary\nAuthor: Rudra Prasad Bhuyan\nDate: 10-10-2026 14:35 IST\n'

In [14]:
import polars as pl

In [15]:
path = r"C:\Users\Rudra\Desktop\rural-financial-inclusion-govt-scheme-recommendation\parquet-data\lev-05\data\lev-05_merged.parquet"
pdf = pl.scan_parquet(path)

In [16]:
pdf.collect_schema()

Schema([('Survey_Name', String),
        ('Year', String),
        ('FSU_Serial_No', String),
        ('Sector', String),
        ('State', String),
        ('NSS_Region', String),
        ('District', String),
        ('Stratum', String),
        ('Sub_stratum', String),
        ('Panel', String),
        ('Sub_sample', String),
        ('FOD_Sub_Region', String),
        ('Sample_SU_No', String),
        ('Sample_Sub_Division_No', String),
        ('Second_Stage_Stratum_No', String),
        ('Sample_Household_No', String),
        ('Questionnaire_No', String),
        ('Level', String),
        ('Item_Code', String),
        ('OutOfHome_Consumption_Quantity', Float64),
        ('OutOfHome_Consumption_Value', Float64),
        ('Total_Consumption_Quantity', Float64),
        ('Total_Consumption_Value', Float64),
        ('Source', String),
        ('Multiplier', Int64)])

# Useful Variables

In [17]:
lev_05 = [
    'OutOfHome_Consumption_Quantity',
    'OutOfHome_Consumption_Value',
    'Total_Consumption_Quantity',
    'Total_Consumption_Value',
    'Source',
]


In [18]:
df = pdf.select(lev_05)

In [19]:
df.head(2).collect()

OutOfHome_Consumption_Quantity,OutOfHome_Consumption_Value,Total_Consumption_Quantity,Total_Consumption_Value,Source
f64,f64,f64,f64,str
10.0,250.0,10.0,250.0,"""2"""
null,null,0.5,160.0,"""1"""


In [20]:
df = df.with_columns(
    [pl.col(col).cast(pl.Int32, strict=False) for col in lev_05]
)

In [21]:
unique_counts = df.select(pl.all().n_unique()).collect()
unique_counts

OutOfHome_Consumption_Quantity,OutOfHome_Consumption_Value,Total_Consumption_Quantity,Total_Consumption_Value,Source
u32,u32,u32,u32,u32
341,2174,1578,4146,9


# Logic

In [22]:
categorical_cols = []
numerical_cols = []

for col in df.columns:
    if unique_counts[col][0] <13:
        categorical_cols.append(col)
    else:
        numerical_cols.append(col)


C:\Users\Rudra\AppData\Local\Temp\ipykernel_23664\2508435801.py:4: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  for col in df.columns:


# Numerical Columns

In [23]:
stats = df.select(numerical_cols).describe()
with pl.Config(tbl_rows=-1, tbl_cols=-1):
    display(stats.to_pandas().T)

,0,1,2,3,4,5,6,7,8
statistic,count,null_count,mean,std,min,25%,50%,75%,max
OutOfHome_Consumption_Quantity,884898.0,24623976.0,15.946127,43.542446,0.0,0.0,2.0,15.0,1450.0
OutOfHome_Consumption_Value,1083592.0,24425282.0,243.387611,399.523818,0.0,15.0,64.0,320.0,30000.0
Total_Consumption_Quantity,22710366.0,2798508.0,27.113932,89.778244,0.0,0.0,1.0,15.0,36000.0
Total_Consumption_Value,25048502.0,460372.0,126.399191,242.847491,0.0,20.0,45.0,123.0,61600.0


# Categorical Columns

In [24]:
for col in categorical_cols:
    print(col)
    counts = df.select(pl.col(col).value_counts(sort=True)).collect()
    
    with pl.Config(tbl_rows=-1, tbl_cols=-1):
        display(counts.unnest(col))

Source


Source,count
i32,u32
1,18969926
null,5874586
2,506344
4,50476
7,33958
6,30370
3,23614
9,13470
5,6130
